# Etapa 3 — Modelagem, validação Walk-Forward e avaliação

Treina e avalia os cinco algoritmos nos 18 folds da validação Walk-Forward (9 por era),
compara-os com um classificador de referência e escolhe o algoritmo levado à explicabilidade
(etapa 4).

| | Arquivo | Conteúdo |
|---|---|---|
| Entrada | `dados/processados/features_jogos.parquet`, `folds_walkforward.csv` | saídas da etapa 2 |
| Saída | `resultados/metricas_folds.csv` | medidas de cada algoritmo em cada fold |
| Saída | `resultados/metricas_era.csv` | médias por era |
| Saída | `resultados/previsoes.parquet` | probabilidade e classe prevista para cada partida de teste |
| Saída | `resultados/hiperparametros.csv`, `optuna_tentativas.csv` | resultado de cada busca e de cada tentativa |
| Saída | `modelos/<era>_<algoritmo>.joblib` | modelo final de cada era, usado na etapa 4 |
| Saída | `resultados/metadados_modelagem.json` | configuração desta execução |

**Regra central: nada é ajustado fora do fold.** Imputação, padronização e busca de
hiperparâmetros usam apenas as temporadas de treino de cada fold. Se a padronização fosse
ajustada na era inteira, por exemplo, a média e o desvio usados no treino já conteriam
informação das temporadas de teste.

A execução completa leva cerca de 80 minutos.

## 1. Configuração

- **Validação interna da busca.** Para escolher os hiperparâmetros sem tocar no teste, o treino
  de cada fold é dividido em quatro blocos em ordem cronológica (`TimeSeriesSplit` com 3
  partições): treina-se no 1º bloco e valida-se no 2º, treina-se nos dois primeiros e valida-se
  no 3º, treina-se nos três e valida-se no 4º. A nota de cada configuração é a média das três
  validações.
- **Tentativas por algoritmo.** Mais tentativas onde o espaço de busca tem configurações ruins
  (SVM, árvore de decisão) e menos onde o desempenho varia pouco entre configurações (floresta
  e *boosting*). A seção 9 mede essa diferença.
- **Contenção de custo, sempre dentro do treino do fold.** A busca do SVM usa no máximo as
  6.000 partidas mais recentes do treino, porque seu custo cresce com o quadrado do número de
  partidas; a configuração vencedora é reajustada no treino completo. Os conjuntos de árvores
  usam 150 árvores durante a busca e 400 no ajuste final.
- **Sem reamostragem nem peso de classe.** A vitória do mandante fica em torno de 60% dos
  jogos, o que não é desbalanceamento severo, e gerar partidas sintéticas criaria históricos de
  equipes que nunca aconteceram.

In [1]:
from collections import Counter
from datetime import datetime
from pathlib import Path
import json, time, warnings

import numpy as np
import pandas as pd
import optuna
from joblib import dump
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
pd.set_option("display.width", 220)

DIR_DADOS = Path("dados/processados")
DIR_RESULTADOS = Path("resultados")
DIR_MODELOS = Path("modelos")

SEMENTE = 42
SPLITS_INTERNOS = 3
MODELOS = ["arvore_decisao", "svm", "random_forest", "xgboost", "lightgbm"]
N_TRIALS = {"arvore_decisao": 30, "svm": 25, "random_forest": 20,
            "xgboost": 25, "lightgbm": 25}
LIMITE_TUNING = {"svm": 6000}
ESTIMADORES_TUNING = 150

COLS_NAO_FEATURES = {"game_id", "data", "temporada", "era", "alvo",
                     "home_team_id", "away_team_id"}

dados = pd.read_parquet(DIR_DADOS / "features_jogos.parquet")
folds = pd.read_csv(DIR_DADOS / "folds_walkforward.csv")
features = [c for c in dados.columns if c not in COLS_NAO_FEATURES]

print(f"{len(dados):,} partidas | {len(features)} variáveis | {len(folds)} divisões")
print(dados.groupby("era").agg(temporadas=("temporada", "nunique"),
                               partidas=("game_id", "size"),
                               taxa_mandante=("alvo", "mean")).round(3).to_string())

c:\Users\Lucas Alves\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


21,210 partidas | 77 variáveis | 18 divisões
          temporadas  partidas  taxa_mandante
era                                          
Classica          10     10479          0.599
Moderna           10     10731          0.567


## 2. Fluxo de cada algoritmo

Cada algoritmo é encapsulado num `Pipeline` de três passos, ajustado só no treino de cada
fold e apenas aplicado ao teste:

1. **imputação** de valores ausentes pela mediana da coluna;
2. **padronização** (subtrai a média e divide pelo desvio-padrão), só no SVM: os algoritmos de
   árvore comparam valores por ordem e não são afetados pela escala;
3. **classificador**.

Valores fixos fora da busca: 400 árvores na floresta e nos dois *boosting*; no XGBoost e no
LightGBM, a fração de partidas (`subsample`) e de variáveis (`colsample_bytree`) sorteada para
cada árvore. No SVM, `probability=True` ativa a calibração de Platt, que converte a saída do SVM
em probabilidade ajustando uma curva logística sobre ela.

In [2]:
def construir(nome):
    passos = [("imput", SimpleImputer(strategy="median"))]
    if nome == "svm":
        passos += [("escala", StandardScaler()),
                   ("mod", SVC(kernel="rbf", probability=True, cache_size=500,
                               random_state=SEMENTE))]
    elif nome == "arvore_decisao":
        passos += [("mod", DecisionTreeClassifier(random_state=SEMENTE))]
    elif nome == "random_forest":
        passos += [("mod", RandomForestClassifier(n_estimators=400, n_jobs=1,
                                                  random_state=SEMENTE))]
    elif nome == "xgboost":
        passos += [("mod", XGBClassifier(n_estimators=400, objective="binary:logistic",
                                         eval_metric="logloss", tree_method="hist",
                                         subsample=0.7, colsample_bytree=0.6,
                                         n_jobs=1, random_state=SEMENTE))]
    elif nome == "lightgbm":
        passos += [("mod", LGBMClassifier(n_estimators=400, n_jobs=1, verbose=-1,
                                          subsample=0.8, subsample_freq=1,
                                          colsample_bytree=0.6, random_state=SEMENTE))]
    return Pipeline(passos)


for nome in MODELOS:
    print(f"{nome:15s}", " -> ".join(type(p).__name__ for _, p in construir(nome).steps))

arvore_decisao  SimpleImputer -> DecisionTreeClassifier
svm             SimpleImputer -> StandardScaler -> SVC
random_forest   SimpleImputer -> RandomForestClassifier
xgboost         SimpleImputer -> XGBClassifier
lightgbm        SimpleImputer -> LGBMClassifier


## 3. Espaços de busca

Intervalos contínuos, em escala logarítmica quando o efeito do hiperparâmetro é multiplicativo,
e ramos condicionais (o `gamma` do SVM pode ser automático ou um valor; a profundidade da árvore
pode ser livre ou limitada). Cada função devolve os parâmetros já com o prefixo `mod__` do
`Pipeline`.

- **Árvore de decisão**: sem poda, uma árvore isolada memoriza o treino; por isso a busca inclui
  o parâmetro de poda por custo-complexidade (`ccp_alpha`), além dos tamanhos mínimos de folha
  e de divisão.
- **SVM**: regularização `C` e largura do núcleo `gamma`.
- **Random Forest**: só o tamanho mínimo da folha, que controla a profundidade efetiva das
  árvores, e o número de variáveis sorteadas em cada divisão de nó.
- **XGBoost e LightGBM**: capacidade de cada árvore (profundidade ou número de folhas), taxa de
  aprendizado e tamanho mínimo da folha.

In [3]:
def espaco_svm(t):
    p = {"mod__C": t.suggest_float("C", 1e-2, 1e2, log=True)}
    p["mod__gamma"] = ("scale" if t.suggest_categorical("gamma_modo",
                                                        ["scale", "valor"]) == "scale"
                       else t.suggest_float("gamma", 1e-4, 1e0, log=True))
    return p


def espaco_arvore_decisao(t):
    return {
        "mod__criterion": t.suggest_categorical("criterion", ["gini", "entropy"]),
        "mod__min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 200, log=True),
        "mod__min_samples_split": t.suggest_int("min_samples_split", 2, 60, log=True),
        "mod__ccp_alpha": t.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
        "mod__max_depth": (None if t.suggest_categorical("profundidade",
                                                         ["livre", "limitada"]) == "livre"
                           else t.suggest_int("max_depth", 2, 20)),
    }


def espaco_random_forest(t):
    return {
        "mod__min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 60, log=True),
        "mod__max_features": ("sqrt" if t.suggest_categorical("max_features_modo",
                                                              ["sqrt", "fracao"]) == "sqrt"
                              else t.suggest_float("max_features", 0.05, 0.8)),
    }


def espaco_xgboost(t):
    return {
        "mod__max_depth": t.suggest_int("max_depth", 2, 9),
        "mod__learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "mod__min_child_weight": t.suggest_int("min_child_weight", 1, 30, log=True),
    }


def espaco_lightgbm(t):
    return {
        "mod__num_leaves": t.suggest_int("num_leaves", 7, 127, log=True),
        "mod__learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "mod__min_child_samples": t.suggest_int("min_child_samples", 5, 120, log=True),
    }


ESPACOS = {"svm": espaco_svm, "arvore_decisao": espaco_arvore_decisao,
           "random_forest": espaco_random_forest, "xgboost": espaco_xgboost,
           "lightgbm": espaco_lightgbm}

exemplo = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=SEMENTE))
for nome in MODELOS:
    buscados = [k.removeprefix("mod__") for k in ESPACOS[nome](exemplo.ask())]
    print(f"{nome:15s} {N_TRIALS[nome]} tentativas por divisão | {', '.join(buscados)}")

arvore_decisao  30 tentativas por divisão | criterion, min_samples_leaf, min_samples_split, ccp_alpha, max_depth
svm             25 tentativas por divisão | C, gamma
random_forest   20 tentativas por divisão | min_samples_leaf, max_features
xgboost         25 tentativas por divisão | max_depth, learning_rate, min_child_weight
lightgbm        25 tentativas por divisão | num_leaves, learning_rate, min_child_samples


## 4. Busca de hiperparâmetros e medidas de desempenho

**Busca.** Usa a biblioteca Optuna com o amostrador TPE (*Tree-structured Parzen Estimator*):
cada tentativa avalia uma configuração, e o TPE usa as tentativas anteriores para propor a
próxima numa região promissora do espaço, em vez de sortear às cegas. Na variante
multivariada, os hiperparâmetros são modelados em conjunto. A **poda pela mediana**
(`MedianPruner`) interrompe uma tentativa quando, depois da segunda partição interna, sua nota
parcial fica abaixo da mediana das tentativas anteriores no mesmo ponto; as cinco primeiras
tentativas nunca são podadas.

**Objetivo da busca: a AUC** (área sob a curva ROC), a probabilidade de o modelo atribuir
chance maior de vitória do mandante a uma partida que o mandante venceu do que a uma que ele
perdeu. Vale 0,5 para um modelo sem poder de discriminação e 1 para um modelo perfeito, e não
depende de um limiar de decisão nem da proporção de vitórias do mandante, que muda entre as
eras. A configuração vencedora é reajustada no treino completo da divisão.

**Medidas no teste**: acurácia, precisão, revocação (*recall*), F1 e AUC. A acurácia e o F1
são comparados com os do classificador da moda no mesmo teste (`ganho_sobre_moda`,
`ganho_f1`). O F1 da moda é alto (cerca de 0,75) porque ela acerta todas as vitórias do
mandante, a classe positiva; sem essa referência, um F1 de 0,72 pareceria bom e estaria abaixo
da regra trivial.

In [4]:
def probabilidades(est, X):
    # Na busca, o SVM roda sem calibração: usa a função de decisão, suficiente para a
    # AUC, que só depende da ordenação.
    if hasattr(est, "predict_proba"):
        return est.predict_proba(X)[:, 1]
    return est.decision_function(X)


def _pipe_busca(pipe):
    # Versão barata do pipeline, usada apenas durante a busca.
    p = clone(pipe)
    params = p.named_steps["mod"].get_params()
    if "n_estimators" in params:
        p.set_params(mod__n_estimators=ESTIMADORES_TUNING)
    if "probability" in params:
        p.set_params(mod__probability=False)
    if "n_jobs" in params:
        p.set_params(mod__n_jobs=-1)
    return p


def ajustar(pipe, nome, Xtr, ytr):
    limite = LIMITE_TUNING.get(nome)
    Xg, yg = ((Xtr.iloc[-limite:], ytr[-limite:]) if limite and len(Xtr) > limite
              else (Xtr, ytr))
    base_pipe = _pipe_busca(pipe)
    splits = list(TimeSeriesSplit(n_splits=SPLITS_INTERNOS).split(Xg))

    def objetivo(trial):
        cand = clone(base_pipe).set_params(**ESPACOS[nome](trial))
        scores = []
        for passo, (itr, iva) in enumerate(splits):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                cand.fit(Xg.iloc[itr], yg[itr])
            scores.append(roc_auc_score(yg[iva], probabilidades(cand, Xg.iloc[iva])))
            trial.report(float(np.mean(scores)), passo)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))

    estudo = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEMENTE, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    estudo.optimize(objetivo, n_trials=N_TRIALS[nome])

    # FixedTrial reconstrói os parâmetros do Pipeline a partir da melhor tentativa.
    params = ESPACOS[nome](optuna.trial.FixedTrial(estudo.best_params))
    est = clone(pipe).set_params(**params).fit(Xtr, ytr)
    trials = estudo.trials_dataframe(attrs=("number", "value", "state", "duration"))
    return est, params, float(estudo.best_value), len(Xg), trials


def avaliar(y, prob, pred, ac_moda, f1_moda):
    m = {"acuracia": accuracy_score(y, pred),
         "precisao": precision_score(y, pred, zero_division=0),
         "recall": recall_score(y, pred, zero_division=0),
         "f1": f1_score(y, pred, zero_division=0),
         "acuracia_moda": ac_moda, "f1_moda": f1_moda}
    m["ganho_sobre_moda"] = m["acuracia"] - ac_moda
    m["ganho_f1"] = m["f1"] - f1_moda
    if prob is not None:                       # a moda não produz probabilidade
        m["auc"] = roc_auc_score(y, prob)
    return m


print("funções de busca e avaliação definidas")

funções de busca e avaliação definidas


## 5. Validação Walk-Forward

A divisão *k* treina nas temporadas 1 a *k* da era e testa na temporada *k* + 1. Em cada
divisão:

1. o **classificador da moda** (`DummyClassifier(strategy="most_frequent")`) aprende a classe
   mais frequente no treino e a repete em todo o teste. É a referência: modelo que não o supera
   não está prevendo nada. Ele é reajustado em cada divisão porque a vantagem de jogar em casa
   varia de temporada para temporada; a classe aprendida é impressa para mostrar que é sempre a
   vitória do mandante;
2. cada algoritmo passa pela busca, é reajustado no treino completo e avaliado no teste.

In [5]:
metricas, previsoes, hiper, tentativas = [], [], [], []

for era in sorted(dados["era"].unique()):
    print(f"\n{'=' * 70}\nERA {era.upper()}\n{'=' * 70}")
    d = dados[dados["era"] == era].sort_values(["data", "game_id"])

    for _, f in folds[folds["era"] == era].sort_values("fold").iterrows():
        treino = d[d["temporada"] <= f["treino_ate"]]
        teste = d[d["temporada"] == f["teste"]]
        Xtr, ytr = treino[features], treino["alvo"].to_numpy()
        Xte, yte = teste[features], teste["alvo"].to_numpy()
        comum = {"era": era, "fold": int(f["fold"]), "teste": int(f["teste"]),
                 "n_treino": len(treino), "n_teste": len(teste),
                 "salto_temporal": int(f["salto_temporal"])}
        print(f"\n{era} divisão {f['fold']} (treino {f['treino_de']}-{f['treino_ate']} -> "
              f"teste {f['teste']}) | treino {len(treino):,} / teste {len(teste):,}")

        moda = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
        pred_moda = moda.predict(Xte)
        ac_moda = accuracy_score(yte, pred_moda)
        f1_moda = f1_score(yte, pred_moda, zero_division=0)
        metricas.append({**comum, "modelo": "moda", "segundos": 0.0,
                         **avaliar(yte, None, pred_moda, ac_moda, f1_moda)})
        print(f"   {'moda':14s} ac {ac_moda:.3f} (classe aprendida: {int(pred_moda[0])} = "
              f"{'vitória' if int(pred_moda[0]) == 1 else 'derrota'} do mandante)")

        for nome in MODELOS:
            t0 = time.time()
            est, params, score, n_tuning, trials = ajustar(construir(nome), nome, Xtr, ytr)
            prob, pred = probabilidades(est, Xte), est.predict(Xte)
            met = avaliar(yte, prob, pred, ac_moda, f1_moda)
            seg = time.time() - t0
            print(f"   {nome:14s} ac {met['acuracia']:.3f} (ganho {met['ganho_sobre_moda']:+.3f})"
                  f"  f1 {met['f1']:.3f}  auc {met['auc']:.3f}  [{seg:.0f}s]")

            metricas.append({**comum, "modelo": nome, "segundos": seg, **met})
            hiper.append({**comum, "modelo": nome, "params": params,
                          "score_interno": score, "n_tuning": n_tuning,
                          "tentativas": len(trials),
                          "podadas": int((trials["state"] == "PRUNED").sum())})
            tentativas.append(trials.assign(era=era, fold=int(f["fold"]), modelo=nome))
            previsoes.append(pd.DataFrame({
                "game_id": teste["game_id"].to_numpy(), "era": era,
                "fold": int(f["fold"]), "temporada": int(f["teste"]), "modelo": nome,
                "alvo": yte, "prob": prob, "pred": pred}))

metricas = pd.DataFrame(metricas)
hiper = pd.DataFrame(hiper)
tentativas = pd.concat(tentativas, ignore_index=True)
previsoes = pd.concat(previsoes, ignore_index=True)
print(f"\nconcluído: {len(metricas)} avaliações em {metricas['segundos'].sum() / 60:.0f} minutos")


ERA CLASSICA

Classica divisão 1 (treino 2005-2005 -> teste 2006) | treino 1,069 / teste 1,070
   moda           ac 0.600 (classe aprendida: 1 = vitória do mandante)
   arvore_decisao ac 0.605 (ganho +0.005)  f1 0.681  auc 0.641  [3s]
   svm            ac 0.636 (ganho +0.036)  f1 0.722  auc 0.671  [3s]
   random_forest  ac 0.651 (ganho +0.051)  f1 0.722  auc 0.688  [34s]
   xgboost        ac 0.631 (ganho +0.031)  f1 0.705  auc 0.661  [47s]
   lightgbm       ac 0.632 (ganho +0.032)  f1 0.702  auc 0.670  [15s]

Classica divisão 2 (treino 2005-2006 -> teste 2007) | treino 2,139 / teste 1,071
   moda           ac 0.588 (classe aprendida: 1 = vitória do mandante)
   arvore_decisao ac 0.627 (ganho +0.039)  f1 0.736  auc 0.657  [5s]
   svm            ac 0.588 (ganho +0.000)  f1 0.741  auc 0.677  [14s]
   random_forest  ac 0.641 (ganho +0.053)  f1 0.718  auc 0.674  [49s]
   xgboost        ac 0.635 (ganho +0.047)  f1 0.713  auc 0.668  [53s]
   lightgbm       ac 0.641 (ganho +0.053)  f1 0.715  

## 6. Desempenho por era

Média dos nove folds de cada era: cada temporada de teste pesa o mesmo. A linha `moda` é o
classificador de referência; como ele prevê sempre a vitória do mandante, sua acurácia é a taxa
de vitórias em casa, sua revocação é 1 e ele não produz AUC.

In [6]:
colunas = ["acuracia", "precisao", "recall", "f1", "auc", "acuracia_moda",
           "ganho_sobre_moda", "f1_moda", "ganho_f1"]
g = metricas.groupby(["era", "modelo"])
agregado = g[colunas].mean().add_suffix("_media")
agregado["acuracia_dp"] = g["acuracia"].std()
agregado["auc_dp"] = g["auc"].std()
agregado["folds"] = g.size()
agregado = agregado.reset_index()

tabela = agregado.set_index(["era", "modelo"])[
    ["acuracia_media", "precisao_media", "recall_media", "f1_media", "auc_media",
     "ganho_sobre_moda_media", "ganho_f1_media", "acuracia_dp", "auc_dp"]]
tabela.columns = ["acuracia", "precisao", "recall", "f1", "auc",
                  "ganho_acuracia", "ganho_f1", "dp_acuracia", "dp_auc"]
print(tabela.sort_values(["era", "auc"], ascending=[True, False]).round(4).to_string())

                         acuracia  precisao  recall      f1     auc  ganho_acuracia  ganho_f1  dp_acuracia  dp_auc
era      modelo                                                                                                   
Classica svm               0.6546    0.6655  0.8694  0.7504  0.7166          0.0573    0.0026       0.0412  0.0290
         random_forest     0.6737    0.6990  0.7971  0.7446  0.7164          0.0763   -0.0032       0.0255  0.0259
         xgboost           0.6703    0.7002  0.7846  0.7397  0.7142          0.0730   -0.0081       0.0273  0.0318
         lightgbm          0.6697    0.7005  0.7814  0.7385  0.7142          0.0724   -0.0093       0.0258  0.0294
         arvore_decisao    0.6548    0.6913  0.7659  0.7250  0.6931          0.0574   -0.0228       0.0293  0.0303
         moda              0.5973    0.5973  1.0000  0.7478     NaN          0.0000    0.0000       0.0163     NaN
Moderna  svm               0.6548    0.6581  0.8141  0.7276  0.7034          0.0

### Superação da moda fold a fold

A média pode esconder folds em que o modelo empata com a regra trivial ou fica abaixo dela.
A tabela conta em quantos dos nove folds cada algoritmo supera a moda em acurácia e mostra o
menor ganho observado.

In [7]:
modelos_reais = metricas[metricas["modelo"] != "moda"]
print(modelos_reais.groupby(["era", "modelo"])
      .agg(divisoes=("fold", "size"),
           acima_da_moda=("ganho_sobre_moda", lambda s: int((s > 0).sum())),
           ganho_minimo=("ganho_sobre_moda", "min"),
           ganho_medio=("ganho_sobre_moda", "mean"))
      .round(4).to_string())

                         divisoes  acima_da_moda  ganho_minimo  ganho_medio
era      modelo                                                            
Classica arvore_decisao         9              9        0.0047       0.0574
         lightgbm               9              9        0.0318       0.0724
         random_forest          9              9        0.0514       0.0763
         svm                    9              7        0.0000       0.0573
         xgboost                9              9        0.0308       0.0730
Moderna  arvore_decisao         9              9        0.0410       0.0853
         lightgbm               9              9        0.0429       0.0871
         random_forest          9              9        0.0457       0.0886
         svm                    9              9        0.0475       0.0883
         xgboost                9              9        0.0447       0.0908


### Folds com tratamento à parte

Dois folds têm teste atípico: a que testa em 2012, temporada reduzida pelo *lockout*, e a
que treina até 2019 e testa em 2022, com três anos entre treino e teste por causa da exclusão
de 2020 e 2021. Se o desempenho nelas ficar dentro da variação dos demais da mesma
era, nenhuma das duas situações prejudicou a generalização.

In [8]:
especiais = {2012: "2012 (lockout)", 2022: "2022 (salto de 3 anos)"}
demais = modelos_reais[~modelos_reais["teste"].isin(especiais)]
faixa = demais.groupby(["era", "modelo"]).agg(
    auc_min_demais=("auc", "min"), auc_max_demais=("auc", "max"),
    acuracia_min_demais=("acuracia", "min"), acuracia_max_demais=("acuracia", "max"))
tab = (modelos_reais[modelos_reais["teste"].isin(especiais)]
       .assign(teste=lambda x: x["teste"].map(especiais))
       .set_index(["era", "modelo"])[["teste", "auc", "acuracia"]]
       .join(faixa))
print(tab.round(3).to_string())

                                          teste    auc  acuracia  auc_min_demais  auc_max_demais  acuracia_min_demais  acuracia_max_demais
era      modelo                                                                                                                           
Classica arvore_decisao          2012 (lockout)  0.666     0.624           0.641           0.719                0.605                0.689
         svm                     2012 (lockout)  0.693     0.638           0.671           0.748                0.588                0.703
         random_forest           2012 (lockout)  0.693     0.650           0.674           0.747                0.641                0.710
         xgboost                 2012 (lockout)  0.697     0.645           0.661           0.748                0.631                0.702
         lightgbm                2012 (lockout)  0.698     0.647           0.667           0.747                0.632                0.712
Moderna  arvore_decisao  20

## 7. Algoritmo levado à explicabilidade

A etapa 4 explica **um** algoritmo nas duas eras. A escolha segue dois critérios, nesta ordem:

1. **Compatibilidade com o explicador exato de árvores.** O SHAP calcula as contribuições de
   forma exata e em minutos para modelos baseados em árvores (`TreeExplainer`). Para o SVM só
   existe o explicador agnóstico ao modelo (`KernelExplainer`), aproximado e inviável em cerca de
   10 mil partidas × 77 variáveis. O SVM sai da disputa por isso, e não por desempenho.
2. **Maior AUC na validação interna** da busca, entre os quatro restantes. A AUC de teste não
   participa da escolha: ela é a medida de desempenho reportada, e com os algoritmos separados
   por poucos milésimos, escolher o máximo dela enviesaria essa estimativa para cima.

In [9]:
MODELOS_ARVORE = ["arvore_decisao", "random_forest", "xgboost", "lightgbm"]

interno = (hiper.groupby(["modelo", "era"])["score_interno"].mean().unstack()
           .assign(media=lambda x: x.mean(axis=1))
           .sort_values("media", ascending=False))
desempenho = modelos_reais.groupby("modelo")["auc"].agg(["mean", "std"]).add_prefix("teste_")
print("AUC da validação interna (critério da escolha) e AUC de teste:")
print(interno.join(desempenho).round(5).to_string())

algoritmo = [m for m in interno.index if m in MODELOS_ARVORE][0]
custo = interno.loc[algoritmo, "media"] - interno["media"].iloc[0]
print(f"\nalgoritmo levado à etapa 4: {algoritmo}")
print(f"  diferença para o primeiro da validação interna ({interno.index[0]}): {custo:+.5f}, "
      f"contra desvio-padrão de {desempenho['teste_std'].mean():.4f} da AUC entre divisões")

AUC da validação interna (critério da escolha) e AUC de teste:
                Classica  Moderna    media  teste_mean  teste_std
modelo                                                           
svm              0.71795  0.70494  0.71145     0.70999    0.02940
random_forest    0.71414  0.70592  0.71003     0.70981    0.02976
xgboost          0.71422  0.70533  0.70978     0.70845    0.03149
lightgbm         0.71399  0.70387  0.70893     0.70764    0.03092
arvore_decisao   0.69449  0.68412  0.68931     0.68656    0.03333

algoritmo levado à etapa 4: random_forest
  diferença para o primeiro da validação interna (svm): -0.00141, contra desvio-padrão de 0.0310 da AUC entre divisões


## 8. Comparação pareada entre os algoritmos

Os algoritmos ficam muito próximos na média, e o desvio-padrão da AUC entre as temporadas de
teste é de cerca de 0,03. Para saber se alguma diferença resiste a essa variação, a comparação
é **pareada**: em cada fold os dois modelos veem o mesmo treino e o mesmo teste, então a
diferença calculada fold a fold elimina o que é comum aos dois, como a dificuldade da
temporada. A tabela compara o algoritmo escolhido com cada um dos demais: diferença média de
AUC, em quantos dos nove folds ele tem a AUC maior, e diferença média de acurácia.

A segunda tabela mostra a **política de decisão** de cada modelo: em que porcentagem das
partidas de teste ele prevê vitória do mandante. Um modelo que prevê mandante com mais
frequência ganha revocação e perde precisão, e isso explica diferenças de acurácia entre
modelos de AUC igual.

In [10]:
referencia = modelos_reais[modelos_reais["modelo"] == algoritmo].set_index(["era", "fold"])
linhas = []
for (era, outro), g in modelos_reais[modelos_reais["modelo"] != algoritmo].groupby(["era", "modelo"]):
    g = g.set_index(["era", "fold"])
    r = referencia.loc[g.index]
    linhas.append({"era": era, "comparacao": f"{algoritmo} - {outro}",
                   "dif_auc": (r["auc"] - g["auc"]).mean(),
                   "divisoes_a_favor_auc": f"{int((r['auc'] > g['auc']).sum())} de {len(g)}",
                   "dif_acuracia": (r["acuracia"] - g["acuracia"]).mean()})
print(pd.DataFrame(linhas).round(4).to_string(index=False))

print("\nPartidas de teste em que o modelo prevê vitória do mandante (%):")
print((previsoes.groupby(["era", "modelo"])["pred"].mean() * 100).unstack().round(1).to_string())

     era                     comparacao  dif_auc divisoes_a_favor_auc  dif_acuracia
Classica random_forest - arvore_decisao   0.0233               9 de 9        0.0189
Classica       random_forest - lightgbm   0.0022               6 de 9        0.0039
Classica            random_forest - svm  -0.0002               4 de 9        0.0191
Classica        random_forest - xgboost   0.0022               3 de 9        0.0033
 Moderna random_forest - arvore_decisao   0.0232               9 de 9        0.0033
 Moderna       random_forest - lightgbm   0.0021               6 de 9        0.0016
 Moderna            random_forest - svm  -0.0002               3 de 9        0.0003
 Moderna        random_forest - xgboost   0.0005               5 de 9       -0.0022

Partidas de teste em que o modelo prevê vitória do mandante (%):
modelo    arvore_decisao  lightgbm  random_forest   svm  xgboost
era                                                             
Classica            66.6      66.6           68.

## 9. Diagnóstico da busca de hiperparâmetros

Três perguntas sobre a busca, cada uma com uma resposta numérica:

- **(a) A busca inflou a estimativa?** Compara a AUC da validação interna com a do teste. Uma
  diferença grande e positiva indicaria hiperparâmetros ajustados ao ruído da validação.
- **(b) Quanto a busca paga?** Compara, dentro de cada busca, a melhor tentativa com a mediana e
  com a pior. Se a melhor supera a mediana por muito menos que a variação entre folds, o
  algoritmo está num platô: quase qualquer configuração serve. A tabela inclui a proporção de
  tentativas interrompidas pela poda.
- **(c) As escolhas são estáveis?** Folds vizinhos compartilham quase todo o treino. Mede-se a
  dispersão de cada hiperparâmetro numérico escolhido entre os folds de uma mesma era e entre
  as médias das duas eras, relativa à média do hiperparâmetro. Dispersão dentro da era maior que
  entre as eras significa que não existe um "hiperparâmetro de cada era": a variação é sorteio
  da busca, o que sustenta usar a mesma configuração nas duas eras (seção 11).

In [11]:
# (a) validação interna x teste
comp = hiper[["era", "fold", "modelo", "score_interno"]].merge(
    modelos_reais[["era", "fold", "modelo", "auc"]], on=["era", "fold", "modelo"])
comp["gap"] = comp["score_interno"] - comp["auc"]
print("(a) AUC da validação interna x AUC do teste (gap > 0 = busca otimista)")
print(comp.groupby("modelo").agg(interno=("score_interno", "mean"), teste=("auc", "mean"),
                                 gap=("gap", "mean")).round(4).to_string())
print(f"    gap médio geral: {comp['gap'].mean():+.4f}")

# (b) ganho da busca dentro de cada estudo, e poda
linhas = []
for (era, fold, modelo), g in tentativas.groupby(["era", "fold", "modelo"]):
    v = g["value"].dropna()
    linhas.append({"modelo": modelo, "melhor": v.max(), "mediana": v.median(), "pior": v.min()})
ganho = pd.DataFrame(linhas).groupby("modelo").mean()
ganho["melhor_menos_mediana"] = ganho["melhor"] - ganho["mediana"]
ganho["melhor_menos_pior"] = ganho["melhor"] - ganho["pior"]
ganho["podadas_%"] = (tentativas.groupby("modelo")["state"]
                      .apply(lambda s: (s == "PRUNED").mean() * 100))
print("\n(b) Quanto a busca paga (AUC interna, média das 18 buscas)")
print(ganho.sort_values("melhor_menos_mediana").round(4).to_string())
print(f"    referência: desvio-padrão da AUC de teste entre divisões = "
      f"{modelos_reais.groupby('modelo')['auc'].std().mean():.4f}")

# (c) dispersão dos hiperparâmetros escolhidos: dentro da era x entre eras
print("\n(c) Dispersão dos hiperparâmetros numéricos escolhidos (relativa à média)")
plano = pd.json_normalize(hiper["params"]).assign(
    era=hiper["era"].to_numpy(), modelo=hiper["modelo"].to_numpy())
for modelo, g in plano.groupby("modelo"):
    dentro, entre = [], []
    for c in [c for c in g.columns if c.startswith("mod__")]:
        v = pd.to_numeric(g[c], errors="coerce")
        if v.notna().sum() < 6 or v.nunique() < 3 or not v.abs().mean():
            continue
        escala = v.abs().mean()
        dentro.append(g.assign(v=v).groupby("era")["v"].std().mean() / escala)
        entre.append(g.assign(v=v).groupby("era")["v"].mean().std() / escala)
    print(f"    {modelo:16s} entre divisões da mesma era {np.mean(dentro):.2f} | "
          f"entre eras {np.mean(entre):.2f}")

(a) AUC da validação interna x AUC do teste (gap > 0 = busca otimista)
                interno   teste     gap
modelo                                 
arvore_decisao   0.6893  0.6866  0.0027
lightgbm         0.7089  0.7076  0.0013
random_forest    0.7100  0.7098  0.0002
svm              0.7114  0.7100  0.0015
xgboost          0.7098  0.7084  0.0013
    gap médio geral: +0.0014

(b) Quanto a busca paga (AUC interna, média das 18 buscas)
                melhor  mediana    pior  melhor_menos_mediana  melhor_menos_pior  podadas_%
modelo                                                                                     
random_forest   0.7101   0.7065  0.6945                0.0036             0.0156     6.3889
xgboost         0.7108   0.7039  0.6470                0.0069             0.0638    25.3333
lightgbm        0.7101   0.7014  0.6468                0.0087             0.0633    26.0000
svm             0.7123   0.7027  0.5492                0.0096             0.1631    36.4444
arvore_d

## 10. Jogos decididos na prorrogação

A etapa 1 marcou as partidas que foram ao tempo extra. Essa marcação **não é variável do
modelo**, porque só se conhece depois do jogo; ela serve de controle analítico para mostrar
onde o erro se concentra. Uma partida empatada ao fim do tempo normal é, por definição,
equilibrada, e a informação pré-jogo tem alcance reduzido sobre ela. As previsões usadas são as
do algoritmo escolhido nos testes das 18 divisões (previsões fora da amostra de treino).

In [12]:
ot = (pd.read_parquet(DIR_DADOS / "partidas_times.parquet", columns=["game_id", "prorrogacao"])
      .groupby("game_id")["prorrogacao"].max())

p = previsoes[previsoes["modelo"] == algoritmo].merge(ot, on="game_id")
p["acerto"] = (p["pred"] == p["alvo"]).astype(float)
p["grupo"] = np.where(p["prorrogacao"] == 1, "com prorrogação", "tempo normal")

tab = (p.groupby(["era", "grupo"])
       .apply(lambda g: pd.Series({
           "partidas": len(g),
           "taxa_mandante": g["alvo"].mean(),
           "acuracia": g["acerto"].mean(),
           "auc": roc_auc_score(g["alvo"], g["prob"]),
       }), include_groups=False))
print(f"{algoritmo}, previsões fora da amostra, por tipo de jogo:")
print(tab.round(3).to_string())
print(f"\nprorrogação em {p.groupby('era')['prorrogacao'].mean().mul(100).round(1).to_dict()} "
      "% das partidas de cada era")

random_forest, previsões fora da amostra, por tipo de jogo:
                          partidas  taxa_mandante  acuracia    auc
era      grupo                                                    
Classica com prorrogação     594.0          0.554     0.579  0.583
         tempo normal       8816.0          0.601     0.681  0.725
Moderna  com prorrogação     508.0          0.553     0.579  0.583
         tempo normal       9151.0          0.567     0.659  0.711

prorrogação em {'Classica': 6.3, 'Moderna': 5.3} % das partidas de cada era


## 11. Modelo final de cada era

Os 18 folds produzem 18 modelos, cada um ajustado num pedaço das temporadas. A etapa 4
precisa de **um modelo por era, ajustado na era inteira**. Esse modelo não estima desempenho
(isso vem só das folds): ele representa como as variáveis são usadas em cada era.

- **Mesmo algoritmo e mesmos hiperparâmetros nas duas eras.** Modelos de capacidades diferentes
  distribuem a importância entre as variáveis de forma diferente; com configurações distintas,
  uma diferença de importância entre eras seria inseparável da diferença entre os modelos.
- **A configuração é o valor central das 18 buscas** do algoritmo escolhido: a mediana para
  hiperparâmetros numéricos e o valor mais frequente para os categóricos. O diagnóstico (c) da
  seção 9 mostra que a configuração de um fold isolada é, em boa parte, sorteio.

In [13]:
def params_consenso(lista):
    saida = {}
    for k in sorted({k for p in lista for k in p}):
        vals = [p[k] for p in lista if k in p]
        num = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
        if len(num) > len(vals) / 2:
            med = float(np.median(num))
            saida[k] = int(round(med)) if all(float(v).is_integer() for v in num) else med
        else:
            alvo = Counter(repr(v) for v in vals).most_common(1)[0][0]
            saida[k] = next(v for v in vals if repr(v) == alvo)
    return saida


h = hiper[hiper["modelo"] == algoritmo]
params = params_consenso(list(h["params"]))
print(f"configuração central das {len(h)} buscas de {algoritmo}, usada nas duas eras:")
for k, v in params.items():
    escolhidos = [p.get(k) for p in h["params"]]
    numericos = [x for x in escolhidos if isinstance(x, (int, float)) and not isinstance(x, bool)]
    categoricos = Counter(x for x in escolhidos if x not in numericos)
    partes = [f"{c!r}: {n}" for c, n in categoricos.most_common()]
    if numericos:
        partes.append(f"numérico de {min(numericos):.4g} a {max(numericos):.4g}: {len(numericos)}")
    print(f"   {k:24s} {v!r:>8}   (escolhas nas divisões: {' | '.join(partes)})")

DIR_MODELOS.mkdir(exist_ok=True)
finais = {}
for era in sorted(dados["era"].unique()):
    d = dados[dados["era"] == era]
    pipe = construir(algoritmo).set_params(**params).fit(d[features], d["alvo"].to_numpy())
    arquivo = DIR_MODELOS / f"{era}_{algoritmo}.joblib"
    dump({"pipeline": pipe, "features": features, "era": era,
          "algoritmo": algoritmo, "params": params}, arquivo)
    finais[era] = str(arquivo)
    print(f"{era}: ajustado em {len(d):,} partidas -> {arquivo}")

configuração central das 18 buscas de random_forest, usada nas duas eras:
   mod__max_features          'sqrt'   (escolhas nas divisões: 'sqrt': 13 | numérico de 0.05472 a 0.7998: 5)
   mod__min_samples_leaf          54   (escolhas nas divisões: numérico de 9 a 59: 18)
Classica: ajustado em 10,479 partidas -> modelos\Classica_random_forest.joblib
Moderna: ajustado em 10,731 partidas -> modelos\Moderna_random_forest.joblib


## 12. Salvar

In [14]:
DIR_RESULTADOS.mkdir(exist_ok=True)
metricas.to_csv(DIR_RESULTADOS / "metricas_folds.csv", index=False)
agregado.to_csv(DIR_RESULTADOS / "metricas_era.csv", index=False)
tentativas.to_csv(DIR_RESULTADOS / "optuna_tentativas.csv", index=False)
previsoes.to_parquet(DIR_RESULTADOS / "previsoes.parquet", index=False)
pd.concat([hiper.drop(columns="params"), pd.json_normalize(hiper["params"])],
          axis=1).to_csv(DIR_RESULTADOS / "hiperparametros.csv", index=False)

(DIR_RESULTADOS / "metadados_modelagem.json").write_text(json.dumps({
    "executado_em": datetime.now().isoformat(timespec="seconds"),
    "semente": SEMENTE, "splits_internos": SPLITS_INTERNOS,
    "otimizador": {"biblioteca": "optuna", "sampler": "TPESampler(multivariate=True)",
                   "pruner": "MedianPruner(n_startup_trials=5, n_warmup_steps=1)",
                   "objetivo": "AUC média do TimeSeriesSplit interno",
                   "n_trials": N_TRIALS},
    "modelos": MODELOS,
    "baseline": "moda: DummyClassifier(strategy='most_frequent') por divisão",
    "n_features": len(features),
    "algoritmo_para_shap": algoritmo,
    "criterio_escolha_algoritmo": ("maior AUC da validação interna entre os modelos "
                                   "explicáveis pelo TreeExplainer"),
    "params_finais": {"origem": "valor central das 18 buscas do Walk-Forward",
                      "iguais_nas_duas_eras": True,
                      "valores": {k: str(v) for k, v in params.items()}},
    "modelos_finais": finais,
}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"resultados em {DIR_RESULTADOS}/ e modelos em {DIR_MODELOS}/")

resultados em resultados/ e modelos em modelos/
